In [ ]:
!pip install datasets pandas
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 8.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import json
import os
import gzip
from datasets import load_dataset
from google.colab import drive

# 1. Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/Colab Notebooks'

config = {
    "mgsm_en": os.path.join(base_path, "mgsm_en.tsv"),
    "mgsm_zh": os.path.join(base_path, "mgsm_zh.tsv"),
    "mgsm_es": os.path.join(base_path, "mgsm_es.tsv"),
    "mkqa_local": os.path.join(base_path, "mkqa.jsonl.gz"),
    "num_samples_per_type": 150
}

def build_final_dataset():
    final_dataset = []

    # --- 2. MGSM  ---
    print("🚀 正在从本地合并 MGSM...")
    try:
        df_en = pd.read_csv(config['mgsm_en'], sep='\t', header=None)
        df_zh = pd.read_csv(config['mgsm_zh'], sep='\t', header=None)
        df_es = pd.read_csv(config['mgsm_es'], sep='\t', header=None)
        limit = min(len(df_en), config['num_samples_per_type'])
        for i in range(limit):
            final_dataset.append({
                "id": f"mgsm_{i:03d}",
                "type": "MGSM",
                "ground_truth": str(df_en.iloc[i, 1]),
                "languages": {
                    "en": {"question": df_en.iloc[i, 0], "prompt": f"Solve this math problem. Answer with only the final number: {df_en.iloc[i, 0]}", "model_output": [], "is_correct": None},
                    "zh": {"question": df_zh.iloc[i, 0], "prompt": f"解决这个数学问题。只回答最终数字：{df_zh.iloc[i, 0]}", "model_output": [], "is_correct": None},
                    "es": {"question": df_es.iloc[i, 0], "prompt": f"Resuelve este problema matemático. Responde solo con el número final: {df_es.iloc[i, 0]}", "model_output": [], "is_correct": None}
                }
            })
        print(f"✅ MGSM complete：{limit} itms")
    except Exception as e: print(f"❌ MGSM error (Check file name): {e}")

    # --- 3. XQuAD (Hugging Face) ---
    print("🚀 getting XQuAD...")
    try:
        x_en = load_dataset("google/xquad", "xquad.en", split="validation")
        x_zh = load_dataset("google/xquad", "xquad.zh", split="validation")
        x_es = load_dataset("google/xquad", "xquad.es", split="validation")
        limit = min(len(x_en), config['num_samples_per_type'])
        for i in range(limit):
            final_dataset.append({
                "id": f"xquad_{i:03d}",
                "type": "XQuAD",
                "context_en": x_en[i]['context'],
                "ground_truth": x_en[i]['answers']['text'][0] if x_en[i]['answers']['text'] else "",
                "languages": {
                    "en": {"question": x_en[i]['question'], "prompt": f"Answer concisely based on the context: {x_en[i]['question']}", "model_output": [], "is_correct": None},
                    "zh": {"question": x_zh[i]['question'], "prompt": f"请根据背景简洁回答：{x_zh[i]['question']}", "model_output": [], "is_correct": None},
                    "es": {"question": x_es[i]['question'], "prompt": f"Responde concisamente según el contexto: {x_es[i]['question']}", "model_output": [], "is_correct": None}
                }
            })
        print(f"✅ XQuAD complete：{limit} items")
    except Exception as e: print(f"❌ XQuAD error: {e}")

    # --- 4.  MKQA (JSONL.GZ) ---
    print(f"🚀 Reading from the local path MKQA: {config['mkqa_local']}...")
    try:
        count = 0

        with gzip.open(config['mkqa_local'], 'rt', encoding='utf-8') as f:
            for line in f:
                if count >= config['num_samples_per_type']: break
                item = json.loads(line)

                queries = item['queries']
                zh_key = 'zh_cn' if 'zh_cn' in queries else 'zh-cn'

                if 'en' in queries and zh_key in queries and 'es' in queries:

                    gt = ""
                    if 'en' in item.get('answers', {}):
                        ans_list = item['answers']['en']
                        if ans_list and 'text' in ans_list[0]:
                            gt = ans_list[0]['text']

                    final_dataset.append({
                        "id": f"mkqa_{count:03d}",
                        "type": "MKQA",
                        "ground_truth": gt,
                        "languages": {
                            "en": {"question": queries['en'], "prompt": f"Answer this factual question concisely: {queries['en']}", "model_output": [], "is_correct": None},
                            "zh": {"question": queries[zh_key], "prompt": f"请简洁回答这个事实性问题：{queries[zh_key]}", "model_output": [], "is_correct": None},
                            "es": {"question": queries['es'], "prompt": f"Responde concisamente a esta pregunta factual: {queries['es']}", "model_output": [], "is_correct": None}
                        }
                    })
                    count += 1
        print(f"✅ MKQA complete：{count} items")
    except Exception as e: print(f"❌ Error report (Check path: {e}")

    return final_dataset

full_data = build_final_dataset()
with open('g24_full_multilingual_data.json', 'w', encoding='utf-8') as f:
    json.dump(full_data, f, ensure_ascii=False, indent=2)

print(f"\n🎉 Success! The full dataset has been generated：g24_full_multilingual_data.json")

🚀 正在从本地合并 MGSM...
✅ MGSM complete：150 itms
🚀 getting XQuAD...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

xquad.en/validation-00000-of-00001.parqu(…):   0%|          | 0.00/212k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/1190 [00:00<?, ? examples/s]

xquad.zh/validation-00000-of-00001.parqu(…):   0%|          | 0.00/206k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/1190 [00:00<?, ? examples/s]

xquad.es/validation-00000-of-00001.parqu(…):   0%|          | 0.00/237k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/1190 [00:00<?, ? examples/s]

✅ XQuAD complete：150 items
🚀 Reading from the local path MKQA: /content/drive/MyDrive/Colab Notebooks/mkqa.jsonl.gz...
✅ MKQA complete：150 items

🎉 Success! The full dataset has been generated：g24_full_multilingual_data.json


In [ ]:
!pip install -q groq

import json
import re
import time
from groq import Groq
from google.colab import drive

# --- 1. Configure Groq ---
GROQ_API_KEY = "****"
client = Groq(api_key=GROQ_API_KEY)

MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"

# --- 2. Load Data ---
input_file = 'g24_full_multilingual_data.json'
output_file = 'g24_final_results_groq.json'

try:
    with open(output_file, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    print("🔄 The current progress has been detected. Continue processing...")
except FileNotFoundError:
    with open(input_file, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    print("🆕 Start a new generation task...")

# --- 3. helper function ---
def extract_number(text):
    nums = re.findall(r"[-+]?\d*\.\d+|\d+", str(text))
    return nums[-1] if nums else text

def generate_with_groq(prompt):
    """A Groq call function with current-limiting protection"""
    # 在 generate_with_groq 函数里打印这个，能看到精准的剩余量
    try:
        completion = client.chat.completions.create(
            model=MODEL_ID,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=512,
            stream=False

        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        if "429" in str(e):
            print("🛑 A Groq call function with current-limiting protection..")
            time.sleep(10)
            return generate_with_groq(prompt)
        print(f"⚠️ error: {e}")
        return f"ERROR: {str(e)}"

# --- 4. Major Loop ---
print(f"🚀 Start the Groq acceleration mode and target the model: {MODEL_ID}")

for i, entry in enumerate(dataset):
    is_done = True
    for lang in ['en', 'zh', 'es']:
        out = entry['languages'][lang].get('model_output', [])
        if len(out) < 3:
            is_done = False
            break
    if is_done: continue
    # 修改循环内部构造 prompt 的逻辑


    # # 然后用这个 full_prompt 去调用 generate_with_groq
    # for lang in ['en', 'zh', 'es']:
    #     prompt = entry['languages'][lang]['prompt']

        existing_outputs = [o for o in entry['languages'][lang].get('model_output', []) if "ERROR" not in str(o)]
        needed = 3 - len(existing_outputs)

        if needed > 0:
            new_outputs = []
            for _ in range(needed):
                res = generate_with_groq(prompt)
                new_outputs.append(res)
                time.sleep(1)

            entry['languages'][lang]['model_output'] = existing_outputs + new_outputs

        if entry['type'] == 'MGSM' and entry['languages'][lang]['model_output']:
            gt = str(entry['ground_truth']).strip()
            pred = extract_number(entry['languages'][lang]['model_output'][0])
            entry['languages'][lang]['is_correct'] = (pred == gt)

    # real time saving
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)

    if i % 10 == 0:
        print(f"📈 进度: {i+1}/{len(dataset)} (ID: {entry['id']} 已存档)")

print(f"🎉 The task has been successfully completed! Data storage : {output_file}")

🔄 The current progress has been detected. Continue processing...
🚀 Start the Groq acceleration mode and target the model: meta-llama/llama-4-scout-17b-16e-instruct
📈 进度: 171/450 (ID: xquad_020 已存档)


KeyboardInterrupt: 

In [ ]:
import json
import re
import time
from groq import Groq
from google.api_core import exceptions

# --- 1. 配置区 ---
GROQ_API_KEY = "****"
client = Groq(api_key=GROQ_API_KEY)
MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"  # 你测试过的模型

input_file = 'g24_full_multilingual_data.json'
output_file = 'g24_final_results_groq - 副本.json'

# --- 2. 加载数据 (断点续传) ---
try:
    with open(output_file, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    print("🔄 检测到现有进度，继续处理...")
except FileNotFoundError:
    with open(input_file, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    print("🆕 开始全新的生成任务...")

# --- 3. 核心工具函数 ---
def extract_number(text):
    nums = re.findall(r"[-+]?\d*\.\d+|\d+", str(text))
    return nums[-1] if nums else text

def generate_with_retry(prompt, max_retries=3):
    """带限流保护的 Groq 调用"""
    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                max_tokens=512
            )
            time.sleep(3) # 基础频率保护 (约 20 RPM)
            return completion.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e):
                print(f"⏳ 限流中，休息 30 秒... (详细: {e})")
                time.sleep(35)
            elif "503" in str(e):
                time.sleep(10)
            else:
                print(f"⚠️ 报错: {e}")
                time.sleep(5)
    return "ERROR"

# --- 4. 主逻辑：流水线作业 ---
print(f"🚀 启动生成引擎，目标：{len(dataset)} 条数据...")

for i, entry in enumerate(dataset):
    # 判定该 ID 是否真正完成 (三语各有3个非ERROR回答)
    is_id_complete = True
    for lang in ['en', 'zh', 'es']:
        out = entry['languages'][lang].get('model_output', [])
        valid_out = [o for o in out if "ERROR" not in str(o)]
        if len(valid_out) < 3:
            is_id_complete = False
            break

    if is_id_complete: continue # 跳过已完成的

    # 开始处理该 ID 的三语任务
    for lang in ['en', 'zh', 'es']:
        context = entry.get('context_en', "")
        question = entry['languages'][lang]['question']

        # 核心：动态构造 Prompt
        if entry['type'] == 'XQuAD' and context:
            # 只有 XQuAD 且有 Context 时注入背景
            if lang == 'en':
                prompt = f"Context: {context}\nQuestion: {question}\nAnswer concisely."
            elif lang == 'zh':
                prompt = f"背景信息: {context}\n问题: {question}\n请根据背景简洁回答。"
            else: # es
                prompt = f"Contexto: {context}\nPregunta: {question}\nResponde concisamente."
        else:
            # MGSM 和 MKQA 使用原始定义的 Prompt
            prompt = entry['languages'][lang]['prompt']

        # 检查是否需要补充回答
        current_out = entry['languages'][lang].get('model_output', [])
        valid_out = [o for o in current_out if "ERROR" not in str(o)]

        needed = 3 - len(valid_out)
        if needed > 0:
            print(f"✍️ 正在处理 {entry['id']} [{lang}]，还需补充 {needed} 条...")
            new_results = []
            for _ in range(needed):
                res = generate_with_retry(prompt)
                if "ERROR" not in res:
                    new_results.append(res)

            # 合并有效结果并存回
            entry['languages'][lang]['model_output'] = valid_out + new_results

        # 针对 MGSM 自动打分
        if entry['type'] == 'MGSM' and entry['languages'][lang]['model_output']:
            gt = str(entry['ground_truth']).strip()
            pred = extract_number(entry['languages'][lang]['model_output'][0])
            entry['languages'][lang]['is_correct'] = (pred == gt)

    # 每一条 ID 处理完立刻保存
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)

    if i % 5 == 0:
        print(f"📈 实时进度: {i+1}/{len(dataset)} (最新完成: {entry['id']})")

print(f"🎉 任务全部完成！最终成果已保存至: {output_file}")

🔄 检测到现有进度，继续处理...
🚀 启动生成引擎，目标：451 条数据...
✍️ 正在处理 xquad_005 [en]，还需补充 3 条...
✍️ 正在处理 xquad_005 [zh]，还需补充 3 条...
✍️ 正在处理 xquad_005 [es]，还需补充 3 条...
📈 实时进度: 156/451 (最新完成: xquad_005)
✍️ 正在处理 xquad_006 [en]，还需补充 3 条...
✍️ 正在处理 xquad_006 [zh]，还需补充 3 条...
✍️ 正在处理 xquad_006 [es]，还需补充 3 条...
✍️ 正在处理 xquad_007 [en]，还需补充 3 条...
✍️ 正在处理 xquad_007 [zh]，还需补充 3 条...
✍️ 正在处理 xquad_007 [es]，还需补充 3 条...
✍️ 正在处理 xquad_008 [en]，还需补充 3 条...
✍️ 正在处理 xquad_008 [zh]，还需补充 3 条...
✍️ 正在处理 xquad_008 [es]，还需补充 3 条...
✍️ 正在处理 xquad_009 [en]，还需补充 3 条...
✍️ 正在处理 xquad_009 [zh]，还需补充 3 条...
✍️ 正在处理 xquad_009 [es]，还需补充 3 条...
✍️ 正在处理 xquad_010 [en]，还需补充 3 条...
✍️ 正在处理 xquad_010 [zh]，还需补充 3 条...
✍️ 正在处理 xquad_010 [es]，还需补充 3 条...
📈 实时进度: 161/451 (最新完成: xquad_010)
✍️ 正在处理 xquad_011 [en]，还需补充 3 条...
✍️ 正在处理 xquad_011 [zh]，还需补充 3 条...
✍️ 正在处理 xquad_011 [es]，还需补充 3 条...
✍️ 正在处理 xquad_012 [en]，还需补充 3 条...
✍️ 正在处理 xquad_012 [zh]，还需补充 3 条...
✍️ 正在处理 xquad_012 [es]，还需补充 3 条...
✍️ 正在处理 xquad_013 [en]，还需补充 3 条...
✍️ 正在处理 xquad_01

KeyboardInterrupt: 